# Generalized vs. Personalized Rolling Seizure Forecasts

This notebook implements the eight-patient comparison requested for the final project. For each included patient $i$:

- $P_i$ is trained on patient $i$'s earlier seizures and tested on the chronologically last seizure(s), as close as possible to an 80/20 event split.
- $G_i$ is trained on the other seven included patients and tested on **all** usable seizures and matched interictal episodes from patient $i$.
- $P_i(i)$ and $G_i(i)$ are reported side by side. Differences are descriptive; no objective optimizes the gap between them.

This is exploratory research code, not a clinical warning system.

## Study design and the shared architecture

The eight comparison patients are `PN00`, `PN05`, `PN06`, `PN09`, `PN10`, `PN12`, `PN13`, and `PN14`. The six patients excluded for insufficient personalized train/test seizure counts are `PN01`, `PN03`, `PN07`, `PN11`, `PN16`, and `PN17`.

Both model families use the project's existing nonlinear **discrete-time hazard model**:

1. Read a causal two-minute EEG context ending at the current landmark.
2. Divide it into 5-second micro-windows.
3. For each selected channel, compute relative/log band power, RMS, line length, and usability features.
4. Aggregate each feature over context using mean, latest value, and slope.
5. A histogram gradient-boosting classifier estimates conditional seizure hazard in each of 60 future 5-second bins.
6. Convert hazards into a coherent probability distribution over onset in the next five minutes plus a no-event probability.

For a given patient $i$, the selected $k=4$ channel schema from $P_i$ is locked and supplied to both $P_i$ and $G_i$. Thus the estimator class, feature definitions, number of features, context, forecast horizon, and evaluation metrics match; the fitting population changes.

**Important interpretation:** sharing a target-specific montage isolates the fitted-population comparison, but it is not a completely cold-start generalized model because the montage came from $P_i$'s training portion. A genuinely zero-personalization deployment analysis would need channels selected without using patient $i$.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd


def locate_script_dir() -> Path:
    for base_dir in (Path.cwd(), *Path.cwd().parents):
        direct = base_dir / "generalized_personalized_comparison.py"
        nested = base_dir / "final_project" / "scripts" / "generalized_personalized_comparison.py"
        if direct.exists():
            return base_dir
        if nested.exists():
            return nested.parent
    raise FileNotFoundError("Could not locate final_project/scripts")


SCRIPT_DIR = locate_script_dir()
PROJECT_DIR = SCRIPT_DIR.parent
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import generalized_personalized_comparison as gp
import personalized_channels_workflow as pc
import rolling_seizure_forecasting as rsf

print("Project:", PROJECT_DIR)
print("Comparison patients:", gp.COMPARISON_PATIENTS)

In [ ]:
# Prespecified experiment settings. Do not tune these using P_i versus G_i differences.
CONTEXT_MINUTES = 2
FORECAST_HORIZON_MINUTES = 5
K_CHANNELS = 4
TEST_FRACTION = 0.20
FORCE_REBUILD_FEATURES = False

# Keep False until all eight personalized summaries are present. Turning this on
# reads raw EDF files, builds/reuses caches, and fits eight generalized models.
RUN_FULL_PIPELINE = False

forecast_config = rsf.ForecastConfig(
    context_seconds=CONTEXT_MINUTES * 60,
    horizon_seconds=FORECAST_HORIZON_MINUTES * 60,
    pre_onset_seconds=FORECAST_HORIZON_MINUTES * 60,
    test_fraction=TEST_FRACTION,
)
forecast_config.validate()

assert forecast_config.bin_seconds == 5
assert forecast_config.n_bins == 60
assert K_CHANNELS == 4
print("Configuration validated:", forecast_config)

## Step 1 — Audit eligible patients and the personalized handoff

This stage must succeed before any model fitting. It verifies the 8+6 partition from the episode manifest and reports which $P_i$ artifacts are complete.

In [ ]:
paths = pc.personalized_paths(SCRIPT_DIR)
manifest = pc.load_manifest(paths, forecast_config)
eligibility = gp.audit_patient_eligibility(manifest)

personalized_summary_path = (
    PROJECT_DIR / "results" / "personalized_channels" / "patient_channel_summary.csv"
)
personalized_summary, pending_personalized = gp.load_personalized_handoff(
    personalized_summary_path
)

display(eligibility)
display(personalized_summary[[
    "patient_id", "n_seizures", "n_train_seizures", "n_test_seizures",
    "selected_channels",
]])
display(pending_personalized)

assert len(eligibility.query("study_role == 'comparison'")) == 8
assert len(eligibility.query("study_role == 'excluded_low_event_count'")) == 6
print(
    f"Personalized handoff: {len(personalized_summary)}/8 complete; "
    f"{len(pending_personalized)} pending."
)

At notebook creation time, complete non-quick personalized summaries exist for five patients: `PN00`, `PN06`, `PN10`, `PN12`, and `PN14`. The production handoff is still required for `PN05`, `PN09`, and `PN13`.

The PN05 quick-mode file is intentionally not mixed into the final comparison because it used reduced iterations, one random baseline repeat, and no swap refinement.

## Step 2 — Verify the matched model specification

Each target-specific model receives 24 compact features per selected channel (`8 feature families × 3 context summaries`). With $k=4$, both $P_i$ and $G_i$ receive 96 EEG features plus three future-time hazard coordinates.

In [ ]:
example_channels = ["CH1", "CH2", "CH3", "CH4"]
example_feature_columns = gp.compact_feature_columns(example_channels)

architecture = pd.Series(
    {
        "estimator": "HistGradientBoostingClassifier discrete hazard",
        "context_minutes": CONTEXT_MINUTES,
        "update_interval_seconds": forecast_config.bin_seconds,
        "forecast_horizon_minutes": FORECAST_HORIZON_MINUTES,
        "future_bins": forecast_config.n_bins,
        "selected_channels": K_CHANNELS,
        "features_per_channel": len(example_feature_columns) // K_CHANNELS,
        "EEG_features": len(example_feature_columns),
        "hazard_time_features": 3,
        "max_leaf_nodes": forecast_config.max_leaf_nodes,
        "learning_rate": forecast_config.learning_rate,
        "L2_regularization": forecast_config.l2_regularization,
    },
    name="value",
)
display(architecture.to_frame())
assert len(example_feature_columns) == 96
assert len(example_feature_columns) + 3 == 99
print("Matched architecture invariants passed.")

## Step 3 — Build or reuse channel-separated rolling features

This is the expensive raw-EEG stage. It builds the same causal landmark representation for all eight patients and caches it under `data/processed/personalized_channels/`. No target-patient rows are used to fit $G_i$.

In [ ]:
feature_data = {}

if RUN_FULL_PIPELINE:
    if not pending_personalized.empty:
        raise RuntimeError(
            "Do not run the final eight-model analysis until every personalized "
            "handoff is present. Pending: "
            + ", ".join(pending_personalized["patient_id"])
        )
    for patient_id in gp.COMPARISON_PATIENTS:
        patient_manifest = manifest.loc[manifest["patient_id"].eq(patient_id)]
        feature_data[patient_id] = pc.build_patient_feature_data(
            patient_manifest,
            paths["feature_cache"],
            forecast_config,
            force=FORCE_REBUILD_FEATURES,
        )
    assert set(feature_data) == set(gp.COMPARISON_PATIENTS)
    print("Feature stage complete for all eight patients.")
else:
    print("Feature build intentionally skipped: RUN_FULL_PIPELINE=False")
    print("Set the flag only after the personalized handoff reports 8/8 complete.")

## Step 4 — Fit the eight $G_i$ models

For each target patient:

- lock that patient's $P_i$ channel schema;
- concatenate aligned landmarks from the other seven patients;
- estimate calibration and the warning threshold using leave-one-training-patient-out predictions only;
- fit the final hazard model on all seven training patients;
- evaluate on all seizure and matched interictal episodes from the target patient.

Channels absent from one training patient are represented as missing values. Training stops if a requested feature is absent from all seven training patients.

In [ ]:
generalized_results = []
coverage_frames = []

if RUN_FULL_PIPELINE:
    for target_patient in gp.COMPARISON_PATIENTS:
        print(f"Fitting G_{target_patient} on the other seven patients...")
        result, coverage = gp.run_generalized_patient(
            feature_data,
            personalized_summary,
            target_patient,
            forecast_config,
        )
        generalized_results.append(result)
        coverage_frames.append(coverage)
        print(
            f"  test seizures={result.test_seizures}; "
            f"AUPRC={result.metrics['auprc']:.3f}; "
            f"Brier={result.metrics['binary_brier']:.3f}"
        )
    assert len(generalized_results) == 8
else:
    print("Eight-model fitting intentionally skipped.")

## Step 5 — Compare $P_i(i)$ and $G_i(i)$ without optimizing the difference

The comparison reports AUPRC, AUROC, binary Brier score, seizure sensitivity, time in warning, false alarms/hour, and median warning lead. Every difference is calculated as `G_i(i) - P_i(i)`.

The test denominators intentionally differ according to the requested design: $G_i(i)$ uses all patient-$i$ seizures, while $P_i(i)$ uses the chronologically held-out approximately 20%. Consequently, this is not an event-paired hypothesis test.

In [ ]:
OUTPUT_DIR = PROJECT_DIR / "results" / "generalized_vs_personalized"

if RUN_FULL_PIPELINE:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    generalized_table = gp.generalized_summary(generalized_results)
    model_values, differences = gp.compare_personalized_generalized(
        personalized_summary,
        generalized_table,
    )
    predictions = pd.concat(
        [result.predictions for result in generalized_results],
        ignore_index=True,
    )
    channel_coverage = pd.concat(coverage_frames, ignore_index=True)

    generalized_table.to_csv(OUTPUT_DIR / "generalized_patient_metrics.csv", index=False)
    model_values.to_csv(OUTPUT_DIR / "personalized_generalized_values.csv", index=False)
    differences.to_csv(OUTPUT_DIR / "generalized_minus_personalized.csv", index=False)
    predictions.to_csv(OUTPUT_DIR / "generalized_landmark_predictions.csv", index=False)
    channel_coverage.to_csv(OUTPUT_DIR / "channel_availability_audit.csv", index=False)

    display(model_values)
    display(differences)
    print("Saved comparison artifacts to", OUTPUT_DIR)
else:
    comparison_status = pd.DataFrame({"patient_id": gp.COMPARISON_PATIENTS})
    completed_ids = set(personalized_summary["patient_id"])
    comparison_status["personalized_handoff"] = comparison_status["patient_id"].map(
        lambda patient_id: "ready" if patient_id in completed_ids else "pending"
    )
    comparison_status["generalized_model"] = "not run"
    display(comparison_status)


## Step 6 — Run the fast structural test suite

These tests use synthetic feature frames, not the EEG results. They verify the patient partition, leakage boundary, missing-channel alignment, handoff handling, and raw difference calculation.

In [ ]:
test_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        str(SCRIPT_DIR / "test_generalized_personalized_comparison.py"),
    ],
    cwd=SCRIPT_DIR,
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
assert test_result.returncode == 0
print("Structural test stage passed.")

## Completion checklist

- [x] Confirm the intended eight comparison patients and six exclusions.
- [x] Define one shared rolling discrete-hazard architecture for $P_i$ and $G_i$.
- [x] Enforce seven-patient training and target-patient-only testing for every $G_i$.
- [x] Make $G_i$ use every target-patient seizure while preserving $P_i$'s chronological ~80/20 split.
- [x] Define descriptive `G - P` comparisons without tuning them.
- [x] Add structural tests and artifact outputs.
- [ ] Receive completed personalized results for `PN05`, `PN09`, and `PN13` from the upstream group member.
- [ ] Set `RUN_FULL_PIPELINE=True`, execute the raw-feature stage, fit all eight $G_i$, and save the final comparison.

Do not interpret a lower or higher difference as evidence of superiority without accounting for the unequal test-seizure denominators and the small number of patient-level comparisons.